In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

In [ ]:
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
EPOCHS = 10
NOISE_FACTOR = 0.5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
train_data = datasets.FashionMNIST(root="./data", train=True, transform=transforms.ToTensor(), download = True)
train_dataloader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)

test_data = datasets.FashionMNIST(root="./data", train=False, transform=transforms.ToTensor(), download = True)
test_dataloader = DataLoader(test_data, batch_size=9, shuffle=False, pin_memory=True)

train_data[0][0].shape

In [ ]:
class DenoisingAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=2, padding=1)
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(in_channels=32, out_channels=16, kernel_size=3, stride=2, padding=1, output_padding=0),
            nn.ReLU(),
            nn.ConvTranspose2d(in_channels=16, out_channels=8, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(in_channels=8, out_channels=1, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )
    

    def forward(self, X):
        # print("In:", X.shape)
        encoded = self.encoder(X)
        # print("Latent:", encoded.shape)
        decoded = self.decoder(encoded)
        # print("Out:", decoded.shape)

        return decoded



In [ ]:
model = DenoisingAutoencoder()
model.to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=LEARNING_RATE)

In [ ]:
for epoch in range(EPOCHS):
    train_loss = 0.0
    for images, _ in train_dataloader:
        images = images.to(DEVICE)

        noisy_images = images + NOISE_FACTOR * torch.randn_like(images)
        noisy_images = torch.clamp(noisy_images, 0., 1.)

        outputs = model(noisy_images)

        loss = criterion(outputs, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
    
    print(f"Epoch [{epoch+1} / {EPOCHS}], Loss: {train_loss/len(train_dataloader):.4f}")

In [ ]:
model.eval()

with torch.no_grad():
    dataiter = iter(test_dataloader)
    images, _ = next(dataiter)
    images = images.to(DEVICE)

    noisy_images = images + NOISE_FACTOR * torch.randn_like(images)
    noisy_images = torch.clamp(noisy_images, 0., 1.)

    outputs = model(noisy_images)

    images = images.cpu().numpy()
    noisy_images = noisy_images.cpu().numpy()
    outputs = outputs.cpu().numpy()

    fig, axes = plt.subplots(nrows=3, ncols=9, figsize=(12, 9))
    for i in range(9):
        axes[0, i].imshow(images[i].reshape(28, 28), cmap='gray')
        axes[0, i].axis("off")
        axes[0, i].set_title("Original")

        axes[1, i].imshow(noisy_images[i].reshape(28, 28), cmap='gray')
        axes[1, i].axis("off")
        axes[1, i].set_title("Noisy")

        axes[2, i].imshow(outputs[i].reshape(28, 28), cmap='gray')
        axes[2, i].axis("off")
        axes[2, i].set_title("Denoised")

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.manifold import TSNE
import numpy as np

viz_dataloader = DataLoader(dataset=test_data, batch_size=1000, shuffle=True)

model.eval()
latent_vectors = []
labels_list = []

with torch.no_grad():
    images, labels = next(iter(viz_dataloader))
    images = images.to(DEVICE)

    latent = model.encoder(images)
    latent = latent.view(latent.size(0), -1)
    latent_vectors = latent.cpu().numpy()
    labels_list = labels.numpy()

tsne = TSNE(n_components=2)
latent_2d = tsne.fit_transform(latent_vectors)
plt.figure(figsize=(10, 8))
scatter = plt.scatter(latent_2d[:, 0], latent_2d[:, 1], c=labels_list, cmap='tab10', alpha=0.8)
plt.colorbar(scatter, ticks=range(10), label='Digit (Label)')
plt.title('Latent Space visualized with t-SNE')
plt.xlabel('Dimension t-SNE 1')
plt.ylabel('Dimension t-SNE 2')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
train_data.class_to_idx

In [ ]:
train_bag_idx = torch.where(train_data.targets == 9)[0]
test_bag_idx = torch.where(test_data.targets == 9)[0]
test_dress_idx = torch.where(test_data.targets == 1)[0]

train_bag_loader = DataLoader(Subset(train_data, train_bag_idx), batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
test_bag_loader = DataLoader(Subset(test_data, test_bag_idx), batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
test_dress_loader = DataLoader(Subset(test_data, test_dress_idx), batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)


In [ ]:
model = DenoisingAutoencoder().to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=LEARNING_RATE)

In [ ]:
for epoch in range(EPOCHS):
    train_loss = 0.0
    for images, _ in train_bag_loader:
        images = images.to(DEVICE)

        outputs = model(images)
        loss = criterion(outputs, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {train_loss/len(train_bag_loader):.4f}")

In [ ]:
def compute_reconstruction_losses(model, dataloader):
    criterion_none = nn.MSELoss(reduction='none')
    losses = []
    model.eval()
    with torch.no_grad():
        for images, _ in dataloader:
            images = images.to(DEVICE)
            outputs = model(images)

            loss_per_image = criterion_none(outputs, images).mean(dim=[1, 2, 3])
            losses.extend(loss_per_image.cpu().numpy())
    
    return np.array(losses)

In [ ]:
losses_bags = compute_reconstruction_losses(model, test_bag_loader)
losses_dresses = compute_reconstruction_losses(model, test_dress_loader)

plt.figure(figsize=(10, 6))

plt.hist(losses_bags, bins=50, alpha=0.6, label='Normal (Boot)', color='blue', density=True)
plt.hist(losses_dresses, bins=50, alpha=0.6, label='Anomalous (Trousers)', color='red', density=True)

THRESHOLD = np.percentile(losses_bags, 95)
plt.axvline(x=THRESHOLD, color='black', linestyle='dashed', linewidth=2, label=f'Decision Threshold: {THRESHOLD}')

plt.xlabel('Reconstruction Error (MSE)')
plt.ylabel('Density (frequency of occurrence)')
plt.title('Anomaly Detection using Autoencoder')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Key Takeaways

1. **The convolutional autoencoder is trained on FashionMNIST boots (class 9),** so it learns to reconstruct the normal class it has seen.
2. **Per-image reconstruction MSE is used as an anomaly score:** dresses (class 1) are treated as anomalies and are expected to have larger errors.
3. **The 95th percentile of boot reconstruction errors sets the threshold.** The histogram shows how well this threshold separates normal and anomalous examples.
4. **The same encoder also provides latent features** that can be projected with t-SNE to inspect how examples are organized.